# Random Forest Training with Polars

This notebook keeps the flow simple:
- load the prepared parquet files
- separate features and target
- validate the numeric training inputs
- train and evaluate the model
- save the model and CSV outputs


## 1. Imports and Config


In [1]:
from pathlib import Path

import polars as pl
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
import joblib

TRAIN_FILE = "../data/train.parquet"
TEST_FILE = "../data/test.parquet"
MODEL_DIR = "../model"
RESULTS_DIR = "../model/results"
TARGET_COLUMN = "readmitted"
RANDOM_SEED = 42
MODEL_NAME = "randomforest"

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

MODEL_FILE = Path(MODEL_DIR) / "randomforest_model.joblib"
RESULTS_FILE = Path(RESULTS_DIR) / "randomforest_results.csv"
METRICS_FILE = Path(RESULTS_DIR) / "randomforest_metrics.csv"


## 2. Load the Dataset


In [2]:
for required_file in [TRAIN_FILE, TEST_FILE]:
    if not Path(required_file).exists():
        raise FileNotFoundError(f"Expected parquet file at: {required_file}")

train_df = pl.read_parquet(TRAIN_FILE)
test_df = pl.read_parquet(TEST_FILE)

print(f"train shape: {train_df.shape}")
print(f"test shape: {test_df.shape}")
train_df.head()


train shape: (71236, 50)
test shape: (30530, 50)


encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
i64,i64,str,str,str,str,i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
261726228,85013370,"""Caucasian""","""Female""","""[70-80)""","""?""",1,1,7,3,"""CM""","""?""",54,0,19,5,0,2,"""428""","""585""","""414""",7,"""None""","""Norm""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""NO"""
111735828,24818796,"""AfricanAmerican""","""Female""","""[70-80)""","""?""",1,3,7,13,"""MC""","""InternalMedicine""",49,5,46,0,0,0,"""518""","""424""","""416""",9,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Up""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""","""<30"""
134945874,65502486,"""Caucasian""","""Male""","""[70-80)""","""?""",1,1,7,4,"""MC""","""Pulmonology""",37,0,20,0,0,0,"""428""","""425""","""493""",8,"""None""","""None""","""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""",""">30"""
274571868,41879529,"""Caucasian""","""Female""","""[40-50)""","""?""",1,1,7,3,"""CP""","""?""",60,0,15,3,0,1,"""250.12""","""585""","""276""",9,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Down""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""","""NO"""
427859552,150796508,"""Caucasian""","""Male""","""[60-70)""","""?""",1,1,7,6,"""HM""","""?""",21,6,14,0,0,0,"""410""","""518""","""427""",9,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",""">30"""


## 3. Separate Features and Target

This notebook uses the prepared train and test parquet files exactly as they come from the upstream split notebook.


In [3]:
if TARGET_COLUMN not in train_df.columns or TARGET_COLUMN not in test_df.columns:
    raise ValueError(f"Both parquet files must include the target column: {TARGET_COLUMN}")

feature_columns = [column_name for column_name in train_df.columns if column_name != TARGET_COLUMN]

X_train = train_df.select(feature_columns)
X_test = test_df.select([column_name for column_name in test_df.columns if column_name != TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN].cast(pl.Int32)
y_test = test_df[TARGET_COLUMN].cast(pl.Int32)

print(feature_columns)
print(X_train.shape)
X_train.head()


InvalidOperationError: conversion from `str` to `i32` failed in column 'readmitted' for 71236 out of 71236 values: ["NO", "<30", … "<30"]

Did not show all failed cases as there were too many.

## 4. Validate the Prepared Features

These notebooks expect the upstream split and feature-selection notebook to output numeric, model-ready feature columns.


In [ ]:
test_feature_columns = [column_name for column_name in test_df.columns if column_name != TARGET_COLUMN]

if feature_columns != test_feature_columns:
    raise ValueError(
        "Train and test parquet files must have the same feature columns in the same order."
    )

non_numeric_columns = sorted(
    set(
        [column_name for column_name, dtype in X_train.schema.items() if not dtype.is_numeric()]
        + [column_name for column_name, dtype in X_test.schema.items() if not dtype.is_numeric()]
    )
)

if non_numeric_columns:
    raise ValueError(
        "The split/feature-selection notebook must output model-ready numeric features. "
        f"Non-numeric columns found: {non_numeric_columns}"
    )

print(f"validated {len(feature_columns)} numeric feature columns")


## 5. Train the Model


In [ ]:
model = RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1)
model.fit(X_train.to_numpy(), y_train.to_numpy())

model


## 6. Evaluate and Save


In [ ]:
probabilities = model.predict_proba(X_test.to_numpy())[:, 1]
predictions = (probabilities >= 0.5).astype(int)
actuals = y_test.to_numpy()

metrics = {
    "model": MODEL_NAME,
    "accuracy": accuracy_score(actuals, predictions),
    "precision": precision_score(actuals, predictions, zero_division=0),
    "recall": recall_score(actuals, predictions, zero_division=0),
    "roc_auc": roc_auc_score(actuals, probabilities),
}

metrics_frame = pl.DataFrame([metrics])
results_frame = test_df.with_columns(
    [
        pl.Series("actual", actuals),
        pl.Series("predicted", predictions),
        pl.Series("probability", probabilities),
        pl.lit(MODEL_NAME).alias("model"),
    ]
)

joblib.dump(model, MODEL_FILE)
metrics_frame.write_csv(METRICS_FILE)
results_frame.write_csv(RESULTS_FILE)

print(f"model saved to: {MODEL_FILE}")
print(f"metrics saved to: {METRICS_FILE}")
print(f"results saved to: {RESULTS_FILE}")
metrics_frame


## 7. Preview Results


In [ ]:
results_frame.head(10)
